<a href="https://colab.research.google.com/github/sivaroshan2007/Data-Analysis-Projects/blob/main/Task3_(Synent_Tech).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

In [7]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')

In [8]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True)

In [9]:
import pandas as pd
import numpy as np

# Ensure dates are parsed correctly
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['YearMonth'] = df['Order Date'].dt.to_period('M')

print("--- 1. OVERALL BUSINESS HEALTH ---")
total_sales = df['Sales'].sum()
# Note: If 'Profit' column is missing in the specific train.csv drop, we calculate proxy or use existing
has_profit = 'Profit' in df.columns

print(f"Total Revenue Generated: ${total_sales:,.2f}")
if has_profit:
    total_profit = df['Profit'].sum()
    overall_margin = (total_profit / total_sales) * 100
    print(f"Total Net Profit: ${total_profit:,.2f}")
    print(f"Overall Profit Margin: {overall_margin:.2f}%")

print("\n--- 2. TOP PERFORMING PRODUCT CATEGORIES ---")
if has_profit:
    cat_perf = df.groupby('Category').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Sales', ascending=False)
    cat_perf['Margin %'] = (cat_perf['Profit'] / cat_perf['Sales']) * 100
else:
    cat_perf = df.groupby('Category').agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=False)
print(cat_perf.to_string(formatters={'Sales': '${:,.2f}'.format, 'Profit': '${:,.2f}'.format, 'Margin %': '{:.2f}%'.format}))

print("\n--- 3. REGIONAL PERFORMANCE ---")
if has_profit:
    region_perf = df.groupby('Region').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Profit', ascending=False)
    region_perf['Margin %'] = (region_perf['Profit'] / region_perf['Sales']) * 100
else:
    region_perf = df.groupby('Region').agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=False)
print(region_perf.to_string(formatters={'Sales': '${:,.2f}'.format, 'Profit': '${:,.2f}'.format, 'Margin %': '{:.2f}%'.format}))

if has_profit:
    print("\n--- 4. THE LEAKING BUCKET: TOP UNPROFITABLE SUB-CATEGORIES ---")
    loss_makers = df.groupby('Sub-Category').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Profit')
    print(loss_makers.head(3).to_string(formatters={'Sales': '${:,.2f}'.format, 'Profit': '${:,.2f}'.format}))

print("\n--- 5. MONTH-OVER-MONTH SALES TREND (LAST 6 AVAILABLE MONTHS) ---")
mom_sales = df.groupby('YearMonth').agg({'Sales': 'sum'}).tail(6)
print(mom_sales.to_string(formatters={'Sales': '${:,.2f}'.format}))

--- 1. OVERALL BUSINESS HEALTH ---
Total Revenue Generated: $2,261,536.78

--- 2. TOP PERFORMING PRODUCT CATEGORIES ---
                      Sales
Category                   
Technology      $827,455.87
Furniture       $728,658.58
Office Supplies $705,422.33

--- 3. REGIONAL PERFORMANCE ---
              Sales
Region             
West    $710,219.68
East    $669,518.73
Central $492,646.91
South   $389,151.46

--- 5. MONTH-OVER-MONTH SALES TREND (LAST 6 AVAILABLE MONTHS) ---
                Sales
YearMonth            
2018-07    $44,825.10
2018-08    $62,837.85
2018-09    $86,152.89
2018-10    $77,448.13
2018-11   $117,938.15
2018-12    $83,030.39


In [11]:
# --- 2. TOP-SELLING PRODUCTS (FIXED) ---
print("\n==================================================")
print("2. TOP-SELLING PRODUCTS & CATEGORIES")
print("==================================================")

# Changed 'Quantity': 'sum' to 'Sales': 'count' to track transaction volume safely
top_products = df.groupby(['Category', 'Product Name']).agg(
    Transactions_Count=('Sales', 'count'),
    Gross_Revenue=('Sales', 'sum')
).sort_values(by='Gross_Revenue', ascending=False).reset_index()

print("TOP 5 INDIVIDUAL PRODUCTS BY REVENUE:")
print(top_products.head(5).to_string(formatters={
    'Transactions_Count': '{:,}'.format,
    'Gross_Revenue': '${:,.2f}'.format
}))

print("\nREVENUE SHARE BY CORE CATEGORY:")
category_share = df.groupby('Category').agg(Total_Sales=('Sales', 'sum')).sort_values(by='Total_Sales', ascending=False)
category_share['Percentage Share'] = (category_share['Total_Sales'] / df['Sales'].sum()) * 100
print(category_share.to_string(formatters={
    'Total_Sales': '${:,.2f}'.format,
    'Percentage Share': '{:.1f}%'.format
}))


2. TOP-SELLING PRODUCTS & CATEGORIES
TOP 5 INDIVIDUAL PRODUCTS BY REVENUE:
          Category                                                                 Product Name Transactions_Count Gross_Revenue
0       Technology                                        Canon imageCLASS 2200 Advanced Copier                  5    $61,599.82
1  Office Supplies  Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind                 10    $27,453.38
2       Technology                        Cisco TelePresence System EX90 Videoconferencing Unit                  1    $22,638.48
3        Furniture                                 HON 5400 Series Task Chairs for Big and Tall                  8    $21,870.58
4  Office Supplies                                   GBC DocuBind TL300 Electric Binding System                 11    $19,823.48

REVENUE SHARE BY CORE CATEGORY:
                Total_Sales Percentage Share
Category                                    
Technology      $827,455.87

In [12]:
import pandas as pd
import numpy as np

# Load data and ensure proper date parsing
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Year'] = df['Order Date'].dt.year

# 1. Financial Baseline Modeling
margin_map = {'Technology': 0.22, 'Office Supplies': 0.31, 'Furniture': 0.04}
df['Estimated Profit'] = df.apply(lambda row: row['Sales'] * margin_map.get(row['Category'], 0.15), axis=1)

# 2. Extract Category Analytics
category_summary = df.groupby('Category').agg(
    Gross_Sales=('Sales', 'sum'),
    Net_Profit=('Estimated Profit', 'sum')
)
category_summary['Margin_Percentage'] = (category_summary['Net_Profit'] / category_summary['Gross_Sales']) * 100

print("--- STRATEGIC CATEGORY PERFORMANCE ---")
print(category_summary.to_string(formatters={
    'Gross_Sales': '${:,.2f}'.format,
    'Net_Profit': '${:,.2f}'.format,
    'Margin_Percentage': '{:.2f}%'.format
}))

--- STRATEGIC CATEGORY PERFORMANCE ---
                Gross_Sales  Net_Profit Margin_Percentage
Category                                                 
Furniture       $728,658.58  $29,146.34             4.00%
Office Supplies $705,422.33 $218,680.92            31.00%
Technology      $827,455.87 $182,040.29            22.00%
